##Gold_Layer


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_adjudication.healthcare_adjudication_gold

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_adjudication.healthcare_adjudication_gold.dim_patient AS

SELECT
patient_id,
patient_name,
gender,
dob,
state

FROM healthcare_adjudication.healthcare_adjudication_silver.silver_patient
WHERE is_current = 'Y'

In [0]:
%sql
select * from healthcare_adjudication.healthcare_adjudication_gold.dim_patient limit 5

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_adjudication.healthcare_adjudication_gold.dim_provider AS

SELECT
provider_id,
provider_name,
specialty,
state

FROM healthcare_adjudication.healthcare_adjudication_silver.silver_provider
WHERE is_current = 'Y'

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_adjudication.healthcare_adjudication_gold.dim_payer AS

SELECT
payer_id,
payer_name,
plan_type

FROM healthcare_adjudication.healthcare_adjudication_silver.silver_payer
WHERE is_current = 'Y'

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_adjudication.healthcare_adjudication_gold.dim_policy AS

SELECT
policy_id,
payer_id,
plan,
coverage_limit,
copay

FROM healthcare_adjudication.healthcare_adjudication_silver.silver_policy
WHERE is_current = 'Y'

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_adjudication.healthcare_adjudication_gold.fact_payments AS

SELECT
payment_id,
claim_id,
payer_id,
payment_amount AS payment_amount,
payment_status

FROM healthcare_adjudication.healthcare_adjudication_silver.silver_payments

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare_adjudication.healthcare_adjudication_gold.fact_claims
USING DELTA
AS

SELECT
    c.claim_id,
    c.patient_id,
    c.provider_id,
    c.policy_id,
    c.claim_date,
    c.status,

    -- total claim cost from claim lines
    SUM(cl.line_amount) AS claim_amount,

    -- total payments made
    COALESCE(SUM(p.payment_amount), 0) AS total_paid,

    -- remaining balance
    SUM(cl.line_amount) - COALESCE(SUM(p.payment_amount), 0) AS remaining_balance

FROM healthcare_adjudication.healthcare_adjudication_silver.silver_claims c

LEFT JOIN healthcare_adjudication.healthcare_adjudication_silver.silver_claim_lines cl
    ON c.claim_id = cl.claim_id

LEFT JOIN healthcare_adjudication.healthcare_adjudication_silver.silver_payments p
    ON c.claim_id = p.claim_id

GROUP BY
    c.claim_id,
    c.patient_id,
    c.provider_id,
    c.policy_id,
    c.claim_date,
    c.status;